# Mr. Chicken + MuseTalk 1.5 Oficial no Kaggle/Colab

Este notebook executa o **MuseTalk 1.5 Oficial** (sincronização labial em tempo real da Tencent Music Lyra Lab) como um microserviço REST FastAPI, compatível com o Mr. Chicken local.


In [ ]:
# 1. Configurações de Ambiente e Pastas
from pathlib import Path
import os, secrets, sys, time

WORK_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
REPO_DIR = WORK_DIR / 'MuseTalk'
OUTPUTS_DIR = WORK_DIR / 'mrchicken_lipsync_outputs'
PORT = int(os.environ.get('LIPSYNC_PORT', '8010'))

PYTHON_EXE = '/opt/conda/bin/python' if os.path.exists('/opt/conda/bin/python') else sys.executable
os.environ['PYTHON'] = PYTHON_EXE

# API Key configurada para segurança (deve bater com o LIPSYNC_API_KEY no Next.js)
API_KEY = "13991059620"
os.environ['LIPSYNC_API_KEY'] = API_KEY
os.environ['MUSETALK_REPO_PATH'] = str(REPO_DIR)
os.environ['MUSETALK_OUTPUTS_DIR'] = str(OUTPUTS_DIR)
os.environ['MUSETALK_TIMEOUT_SECONDS'] = '1800'
os.environ['HF_HOME'] = str(WORK_DIR / 'hf-cache')

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print('WORK_DIR=', WORK_DIR)
print('REPO_DIR=', REPO_DIR)
print('OUTPUTS_DIR=', OUTPUTS_DIR)
print('PORT=', PORT)
print('PYTHON_EXE=', PYTHON_EXE)


In [ ]:
# 2. Clone/update do repositório oficial MuseTalk
import shutil, subprocess, sys, os

giturl = 'https://github.com/TMElyralab/MuseTalk.git'

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    print("Diretório MuseTalk existe mas não é repositório git. Removendo...")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    print("Clonando repositório MuseTalk oficial...")
    subprocess.run(['git', 'clone', giturl, str(REPO_DIR)], check=True)
else:
    print("Repositório MuseTalk já existe. Efetuando git pull...")
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

try:
    print("Verificando/instalando pacotes de sistema necessários (libgl1, ffmpeg)...")
    subprocess.run(['sudo', 'apt-get', 'update', '-y'], check=False)
    subprocess.run(['sudo', 'apt-get', 'install', '-y', 'libgl1', 'ffmpeg'], check=False)
except Exception as e:
    print("Aviso ao tentar atualizar apt:", e)

print("Instalando dependências do Python...")
PYTHON_EXE = os.environ.get('PYTHON') or sys.executable
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)

# Instala requirements.txt do próprio repositório MuseTalk
req_file = REPO_DIR / 'requirements.txt'
if req_file.exists():
    subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-r', str(req_file)], check=True)

# Instala pacotes do ecossistema OpenMMLab
print("Instalando bibliotecas do OpenMMLab (openmim, mmengine, mmcv, mmdet, mmpose)...")
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', 'openmim'], check=True)
subprocess.run([PYTHON_EXE, '-m', 'mim', 'install', 'mmengine', 'mmcv>=2.0.1', 'mmdet>=3.1.0', 'mmpose>=1.1.0'], check=True)

# Instala dependências do microserviço FastAPI
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'python-multipart'], check=True)
print("Dependências configuradas com sucesso!")


In [ ]:
# 3. Download de Pesos Oficiais do MuseTalk 1.5
import subprocess, sys, os
from pathlib import Path

PYTHON_EXE = os.environ.get('PYTHON') or sys.executable
print("Garantindo huggingface_hub...")
subprocess.run([PYTHON_EXE, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub'], check=True)

models_dir = REPO_DIR / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

print("Efetuando download de checkpoints de TMElyralab/MuseTalk do Hugging Face...")
from huggingface_hub import snapshot_download
try:
    snapshot_download(
        repo_id='TMElyralab/MuseTalk',
        local_dir=str(models_dir),
        ignore_patterns=['*.git*', 'README.md']
    )
    print("Pesos baixados e organizados com sucesso!")
except Exception as e:
    print("Erro ao baixar pesos via huggingface_hub:", e)
    raise e


In [ ]:
# 4. Validação da Estrutura dos Pesos
import os
from pathlib import Path

models_dir = REPO_DIR / 'models'
expected_paths = [
    models_dir / "musetalkV15" / "unet.pth",
    models_dir / "musetalkV15" / "musetalk.json",
    models_dir / "sd-vae-ft-mse",
    models_dir / "whisper" / "tiny.pt",
    models_dir / "dwpose",
    models_dir / "face-parse-bisent"
]

print("Estrutura de pesos local:")
all_ok = True
for p in expected_paths:
    status_str = "[OK]" if p.exists() else "[FALHA]"
    if not p.exists():
        all_ok = False
    print(f" {status_str} {p.relative_to(models_dir)}")

if all_ok:
    print("\nEstrutura de modelos validada com sucesso!")
else:
    print("\nAviso: Nem todos os arquivos foram encontrados. Verifique logs acima.")


In [ ]:
# 5. Escrever Arquivos do Microserviço
import base64
from pathlib import Path

app_b64 = "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc2h1dGlsCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwKCmZyb20gZmFzdGFwaSBpbXBvcnQgRGVwZW5kcywgRmFzdEFQSSwgRmlsZSwgRm9ybSwgSGVhZGVyLCBIVFRQRXhjZXB0aW9uLCBSZXF1ZXN0LCBVcGxvYWRGaWxlLCBzdGF0dXMKZnJvbSBmYXN0YXBpLnJlc3BvbnNlcyBpbXBvcnQgRmlsZVJlc3BvbnNlCmZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbCwgQ29uZmlnRGljdCwgRmllbGQKCmZyb20gbXVzZXRhbGtfc2VydmljZSBpbXBvcnQgKAogICAgR1BVVW5hdmFpbGFibGVFcnJvciwKICAgIE11c2VUYWxrRXhlY3V0aW9uRXJyb3IsCiAgICBNdXNlVGFsa1NlcnZpY2UsCiAgICBNdXNlVGFsa1RpbWVvdXRFcnJvciwKKQoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1vcy5nZXRlbnYoIkxPR19MRVZFTCIsICJJTkZPIiksIGZvcm1hdD0iJShhc2N0aW1lKXMgJShsZXZlbG5hbWUpcyBbTElQU1lOQ10gJShtZXNzYWdlKXMiKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigibXJjaGlja2VuLmxpcHN5bmMiKQoKYXBwID0gRmFzdEFQSSh0aXRsZT0iTXJDaGlja2VuIE11c2VUYWxrIExpcC1TeW5jIFNlcnZpY2UiLCB2ZXJzaW9uPSIxLjEuMCIpCnNlcnZpY2UgPSBNdXNlVGFsa1NlcnZpY2UoCiAgICBtb2RlbHNfZGlyPVBhdGgob3MuZ2V0ZW52KCJNVVNFVEFMS19NT0RFTFNfRElSIiwgUGF0aChfX2ZpbGVfXykucGFyZW50IC8gIm1vZGVscyIpKSwKICAgIG91dHB1dHNfZGlyPVBhdGgob3MuZ2V0ZW52KCJNVVNFVEFMS19PVVRQVVRTX0RJUiIsIFBhdGgoX19maWxlX18pLnBhcmVudCAvICJvdXRwdXRzIikpLAopCgoKY2xhc3MgR2VuZXJhdGVSZXF1ZXN0KEJhc2VNb2RlbCk6CiAgICBqb2JfaWQ6IHN0ciA9IEZpZWxkKC4uLiwgYWxpYXM9ImpvYklkIiwgbWluX2xlbmd0aD0xKQogICAgYXZhdGFyX3BhdGg6IHN0ciA9IEZpZWxkKC4uLiwgYWxpYXM9ImF2YXRhclBhdGgiLCBtaW5fbGVuZ3RoPTEpCiAgICBhdWRpb19wYXRoOiBzdHIgPSBGaWVsZCguLi4sIGFsaWFzPSJhdWRpb1BhdGgiLCBtaW5fbGVuZ3RoPTEpCgoKY2xhc3MgR2VuZXJhdGVSZXNwb25zZShCYXNlTW9kZWwpOgogICAgbW9kZWxfY29uZmlnID0gQ29uZmlnRGljdChwb3B1bGF0ZV9ieV9uYW1lPVRydWUpCgogICAgc3VjY2VzczogYm9vbAogICAgcHJvdmlkZXI6IHN0ciA9ICJtdXNldGFsay12MTUiCiAgICB2aWRlb19wYXRoOiBzdHIgPSBGaWVsZCguLi4sIGFsaWFzPSJ2aWRlb1BhdGgiKQogICAgdmlkZW9fdXJsOiBPcHRpb25hbFtzdHJdID0gRmllbGQoZGVmYXVsdD1Ob25lLCBhbGlhcz0idmlkZW9VcmwiKQoKCmNsYXNzIEVycm9yUmVzcG9uc2UoQmFzZU1vZGVsKToKICAgIHN1Y2Nlc3M6IGJvb2wgPSBGYWxzZQogICAgcHJvdmlkZXI6IHN0ciA9ICJtdXNldGFsay12MTUiCiAgICBlcnJvcjogc3RyCiAgICBjb2RlOiBzdHIKICAgIGRldGFpbHM6IE9wdGlvbmFsW3N0cl0gPSBOb25lCgoKZGVmIHZlcmlmeV9hcGlfa2V5KAogICAgYXV0aG9yaXphdGlvbjogT3B0aW9uYWxbc3RyXSA9IEhlYWRlcihkZWZhdWx0PU5vbmUpLAogICAgeF9hcGlfa2V5OiBPcHRpb25hbFtzdHJdID0gSGVhZGVyKGRlZmF1bHQ9Tm9uZSksCikgLT4gTm9uZToKICAgICMgQVBJIGtleSB2ZXJpZmljYXRpb24gbG9naWMKICAgIGV4cGVjdGVkX2tleSA9IG9zLmdldGVudigiTElQU1lOQ19BUElfS0VZIikKICAgIGlmIG5vdCBleHBlY3RlZF9rZXk6CiAgICAgICAgcmV0dXJuCiAgICAgICAgCiAgICBwcm92aWRlZF9rZXkgPSBOb25lCiAgICBpZiB4X2FwaV9rZXk6CiAgICAgICAgcHJvdmlkZWRfa2V5ID0geF9hcGlfa2V5LnN0cmlwKCkKICAgIGVsaWYgYXV0aG9yaXphdGlvbiBhbmQgYXV0aG9yaXphdGlvbi5zdGFydHN3aXRoKCJCZWFyZXIgIik6CiAgICAgICAgcHJvdmlkZWRfa2V5ID0gYXV0aG9yaXphdGlvbltsZW4oIkJlYXJlciAiKTpdLnN0cmlwKCkKICAgICAgICAKICAgIGlmIHByb3ZpZGVkX2tleSAhPSBleHBlY3RlZF9rZXkuc3RyaXAoKToKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKAogICAgICAgICAgICBzdGF0dXNfY29kZT1zdGF0dXMuSFRUUF80MDFfVU5BVVRIT1JJWkVELAogICAgICAgICAgICBkZXRhaWw9ewogICAgICAgICAgICAgICAgInN1Y2Nlc3MiOiBGYWxzZSwKICAgICAgICAgICAgICAgICJwcm92aWRlciI6ICJtdXNldGFsay12MTUiLAogICAgICAgICAgICAgICAgImVycm9yIjogIkNoYXZlIGRlIEFQSSBpbnbDoWxpZGEgb3UgYXVzZW50ZS4iLAogICAgICAgICAgICAgICAgImNvZGUiOiAiVU5BVVRIT1JJWkVEIgogICAgICAgICAgICB9CiAgICAgICAgKQoKCmRlZiBzYWZlX3BhdGhfc2VnbWVudCh2YWx1ZTogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gcmUuc3ViKHIiW15hLXpBLVowLTkuXy1dIiwgIi0iLCB2YWx1ZSlbOjEyMF0gb3IgImpvYiIKCgpkZWYgc2FmZV91cGxvYWRfbmFtZShmaWxlbmFtZTogc3RyIHwgTm9uZSwgZmFsbGJhY2s6IHN0cikgLT4gc3RyOgogICAgc3VmZml4ID0gUGF0aChmaWxlbmFtZSBvciBmYWxsYmFjaykuc3VmZml4Lmxvd2VyKCkKICAgIGlmIHN1ZmZpeCBub3QgaW4geyIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLndlYnAiLCAiLm1wNCIsICIubW92IiwgIi53ZWJtIiwgIi53YXYiLCAiLm1wMyIsICIubTRhIn06CiAgICAgICAgc3VmZml4ID0gUGF0aChmYWxsYmFjaykuc3VmZml4CiAgICByZXR1cm4gZiJ7UGF0aChmYWxsYmFjaykuc3RlbX17c3VmZml4fSIKCgpkZWYgc2F2ZV91cGxvYWQodXBsb2FkOiBVcGxvYWRGaWxlLCBkZXN0aW5hdGlvbjogUGF0aCkgLT4gUGF0aDoKICAgIGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIGRlc3RpbmF0aW9uLm9wZW4oIndiIikgYXMgb3V0cHV0OgogICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaih1cGxvYWQuZmlsZSwgb3V0cHV0KQogICAgcmV0dXJuIGRlc3RpbmF0aW9uCgoKZGVmIGVycm9yX3Jlc3BvbnNlKHN0YXR1c19jb2RlOiBpbnQsIGVycm9yOiBzdHIsIGNvZGU6IHN0ciwgZGV0YWlsczogT3B0aW9uYWxbc3RyXSA9IE5vbmUpIC0+IEhUVFBFeGNlcHRpb246CiAgICByZXR1cm4gSFRUUEV4Y2VwdGlvbigKICAgICAgICBzdGF0dXNfY29kZT1zdGF0dXNfY29kZSwKICAgICAgICBkZXRhaWw9ewogICAgICAgICAgICAic3VjY2VzcyI6IEZhbHNlLAogICAgICAgICAgICAicHJvdmlkZXIiOiAibXVzZXRhbGstdjE1IiwKICAgICAgICAgICAgImVycm9yIjogZXJyb3IsCiAgICAgICAgICAgICJjb2RlIjogY29kZSwKICAgICAgICAgICAgImRldGFpbHMiOiBkZXRhaWxzIG9yICIiCiAgICAgICAgfSwKICAgICkKCgpkZWYgcnVuX2dlbmVyYXRpb24oam9iX2lkOiBzdHIsIGF2YXRhcl9wYXRoOiBQYXRoLCBhdWRpb19wYXRoOiBQYXRoKSAtPiBQYXRoOgogICAgdHJ5OgogICAgICAgIHJldHVybiBzZXJ2aWNlLmdlbmVyYXRlKAogICAgICAgICAgICBqb2JfaWQ9am9iX2lkLAogICAgICAgICAgICBhdmF0YXJfcGF0aD1hdmF0YXJfcGF0aCwKICAgICAgICAgICAgYXVkaW9fcGF0aD1hdWRpb19wYXRoLAogICAgICAgICkKICAgIGV4Y2VwdCBGaWxlTm90Rm91bmRFcnJvciBhcyBleGM6CiAgICAgICAgcmFpc2UgZXJyb3JfcmVzcG9uc2Uoc3RhdHVzLkhUVFBfNDA0X05PVF9GT1VORCwgc3RyKGV4YyksICJGSUxFX05PVF9GT1VORCIpIGZyb20gZXhjCiAgICBleGNlcHQgR1BVVW5hdmFpbGFibGVFcnJvciBhcyBleGM6CiAgICAgICAgcmFpc2UgZXJyb3JfcmVzcG9uc2Uoc3RhdHVzLkhUVFBfNTAzX1NFUlZJQ0VfVU5BVkFJTEFCTEUsIHN0cihleGMpLCAiR1BVX1VOQVZBSUxBQkxFIikgZnJvbSBleGMKICAgIGV4Y2VwdCBNdXNlVGFsa1RpbWVvdXRFcnJvciBhcyBleGM6CiAgICAgICAgcmFpc2UgZXJyb3JfcmVzcG9uc2Uoc3RhdHVzLkhUVFBfNTA0X0dBVEVXQVlfVElNRU9VVCwgc3RyKGV4YyksICJUSU1FT1VUIikgZnJvbSBleGMKICAgIGV4Y2VwdCBNdXNlVGFsa0V4ZWN1dGlvbkVycm9yIGFzIGV4YzoKICAgICAgICBqb2JfZGlyID0gc2VydmljZS5vdXRwdXRzX2RpciAvIGpvYl9pZAogICAgICAgIGVycm9yX2RldGFpbHMgPSAiIgogICAgICAgIHRyeToKICAgICAgICAgICAgZXJyX2xvZ19wYXRoID0gam9iX2RpciAvICJlcnJvci5sb2ciCiAgICAgICAgICAgIGlmIGVycl9sb2dfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgICAgIGVycm9yX2RldGFpbHMgPSBlcnJfbG9nX3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpWy0xMDAwOl0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHN0ZGVycl9sb2dfcGF0aCA9IGpvYl9kaXIgLyAic3RkZXJyLmxvZyIKICAgICAgICAgICAgICAgIGlmIHN0ZGVycl9sb2dfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgICAgICAgICBlcnJvcl9kZXRhaWxzID0gc3RkZXJyX2xvZ19wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKVstMTAwMDpdCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcGFzcwogICAgICAgIHJhaXNlIGVycm9yX3Jlc3BvbnNlKAogICAgICAgICAgICBzdGF0dXNfY29kZT1zdGF0dXMuSFRUUF81MDBfSU5URVJOQUxfU0VSVkVSX0VSUk9SLAogICAgICAgICAgICBlcnJvcj1zdHIoZXhjKSwKICAgICAgICAgICAgY29kZT0iTVVTRVRBTEtfRVJST1IiLAogICAgICAgICAgICBkZXRhaWxzPWVycm9yX2RldGFpbHMgb3Igc3RyKGV4YykKICAgICAgICApIGZyb20gZXhjCgoKQGFwcC5nZXQoIi9oZWFsdGgiKQpkZWYgaGVhbHRoKCkgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICBpbXBvcnRzX3N0YXR1cyA9IHt9CiAgICBjcml0aWNhbF9saWJzID0gWyJ0b3JjaCIsICJkaWZmdXNlcnMiLCAidHJhbnNmb3JtZXJzIiwgIm1tcG9zZSIsICJtbWN2IiwgIm1tZGV0IiwgIm1tZW5naW5lIl0KICAgIGZvciBsaWIgaW4gY3JpdGljYWxfbGliczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIF9faW1wb3J0X18obGliKQogICAgICAgICAgICBpbXBvcnRzX3N0YXR1c1tsaWJdID0gIk9LIgogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvciBhcyBlOgogICAgICAgICAgICBpbXBvcnRzX3N0YXR1c1tsaWJdID0gZiJNaXNzaW5nOiB7c3RyKGUpfSIKICAgICAgICAgICAgCiAgICBncHVfYXZhaWwgPSBzZXJ2aWNlLmdwdV9hdmFpbGFibGUoKQogICAgCiAgICBjb25maWdfZGV0YWlscyA9IHsKICAgICAgICAibXVzZXRhbGtfdmVyc2lvbiI6IG9zLmdldGVudigiTVVTRVRBTEtfVkVSU0lPTiIsICJ2MTUiKSwKICAgICAgICAidW5ldF9tb2RlbF9wYXRoIjogb3MuZ2V0ZW52KCJNVVNFVEFMS19VTkVUX01PREVMX1BBVEgiLCAibW9kZWxzL211c2V0YWxrVjE1L3VuZXQucHRoIiksCiAgICAgICAgInVuZXRfY29uZmlnIjogb3MuZ2V0ZW52KCJNVVNFVEFMS19VTkVUX0NPTkZJRyIsICJtb2RlbHMvbXVzZXRhbGtWMTUvbXVzZXRhbGsuanNvbiIpLAogICAgICAgICJyZXF1aXJlX2dwdSI6IG9zLmdldGVudigiTVVTRVRBTEtfUkVRVUlSRV9HUFUiLCAidHJ1ZSIpLAogICAgICAgICJ0aW1lb3V0X3NlY29uZHMiOiBvcy5nZXRlbnYoIk1VU0VUQUxLX1RJTUVPVVRfU0VDT05EUyIsICIxODAwIiksCiAgICB9CiAgICAKICAgIHJldHVybiB7CiAgICAgICAgInN1Y2Nlc3MiOiBUcnVlLAogICAgICAgICJlbmdpbmUiOiAibXVzZXRhbGstdjE1IiwKICAgICAgICAiZ3B1QXZhaWxhYmxlIjogZ3B1X2F2YWlsLAogICAgICAgICJyZXBvUGF0aCI6IG9zLmdldGVudigiTVVTRVRBTEtfUkVQT19QQVRIIiwgIiIpLAogICAgICAgICJvdXRwdXRzUGF0aCI6IHN0cihzZXJ2aWNlLm91dHB1dHNfZGlyLnJlc29sdmUoKSksCiAgICAgICAgImNvbmZpZyI6IGNvbmZpZ19kZXRhaWxzLAogICAgICAgICJpbXBvcnRzIjogaW1wb3J0c19zdGF0dXMKICAgIH0KCgpAYXBwLnBvc3QoIi9nZW5lcmF0ZSIsIHJlc3BvbnNlX21vZGVsPUdlbmVyYXRlUmVzcG9uc2UsIHJlc3BvbnNlcz17CiAgICA0MDE6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwKICAgIDQwNDogeyJtb2RlbCI6IEVycm9yUmVzcG9uc2V9LAogICAgNTAwOiB7Im1vZGVsIjogRXJyb3JSZXNwb25zZX0sCiAgICA1MDM6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwKICAgIDUwNDogeyJtb2RlbCI6IEVycm9yUmVzcG9uc2V9LAp9KQpkZWYgZ2VuZXJhdGUocmVxdWVzdDogR2VuZXJhdGVSZXF1ZXN0LCBfOiBOb25lID0gRGVwZW5kcyh2ZXJpZnlfYXBpX2tleSkpIC0+IEdlbmVyYXRlUmVzcG9uc2U6CiAgICBsb2dnZXIuaW5mbygiSm9iICVzIHJlY2ViaWRvOiBhdmF0YXI9JXMgYXVkaW89JXMiLCByZXF1ZXN0LmpvYl9pZCwgcmVxdWVzdC5hdmF0YXJfcGF0aCwgcmVxdWVzdC5hdWRpb19wYXRoKQogICAgdmlkZW9fcGF0aCA9IHJ1bl9nZW5lcmF0aW9uKAogICAgICAgIGpvYl9pZD1yZXF1ZXN0LmpvYl9pZCwKICAgICAgICBhdmF0YXJfcGF0aD1QYXRoKHJlcXVlc3QuYXZhdGFyX3BhdGgpLAogICAgICAgIGF1ZGlvX3BhdGg9UGF0aChyZXF1ZXN0LmF1ZGlvX3BhdGgpLAogICAgKQogICAgbG9nZ2VyLmluZm8oIkpvYiAlcyBjb25jbHXDrWRvOiAlcyIsIHJlcXVlc3Quam9iX2lkLCB2aWRlb19wYXRoKQogICAgcmV0dXJuIEdlbmVyYXRlUmVzcG9uc2Uoc3VjY2Vzcz1UcnVlLCBwcm92aWRlcj0ibXVzZXRhbGstdjE1IiwgdmlkZW9fcGF0aD1zdHIodmlkZW9fcGF0aCkpCgoKQGFwcC5wb3N0KCIvZ2VuZXJhdGUtdXBsb2FkIiwgcmVzcG9uc2VfbW9kZWw9R2VuZXJhdGVSZXNwb25zZSwgcmVzcG9uc2VzPXsKICAgIDQwMTogeyJtb2RlbCI6IEVycm9yUmVzcG9uc2V9LAogICAgNDA0OiB7Im1vZGVsIjogRXJyb3JSZXNwb25zZX0sCiAgICA1MDA6IHsibW9kZWwiOiBFcnJvclJlc3BvbnNlfSwKICAgIDUwMzogeyJtb2RlbCI6IEVycm9yUmVzcG9uc2V9LAogICAgNTA0OiB7Im1vZGVsIjogRXJyb3JSZXNwb25zZX0sCn0pCmRlZiBnZW5lcmF0ZV91cGxvYWQoCiAgICByZXF1ZXN0OiBSZXF1ZXN0LAogICAgam9iSWQ6IHN0ciA9IEZvcm0oLi4uLCBtaW5fbGVuZ3RoPTEpLAogICAgYXZhdGFyOiBPcHRpb25hbFtVcGxvYWRGaWxlXSA9IEZpbGUoZGVmYXVsdD1Ob25lKSwKICAgIHZpZGVvOiBPcHRpb25hbFtVcGxvYWRGaWxlXSA9IEZpbGUoZGVmYXVsdD1Ob25lKSwKICAgIGF1ZGlvOiBVcGxvYWRGaWxlID0gRmlsZSguLi4pLAogICAgXzogTm9uZSA9IERlcGVuZHModmVyaWZ5X2FwaV9rZXkpLAopIC0+IEdlbmVyYXRlUmVzcG9uc2U6CiAgICBzYWZlX2pvYl9pZCA9IHNhZmVfcGF0aF9zZWdtZW50KGpvYklkKQogICAgam9iX2RpciA9IHNlcnZpY2Uub3V0cHV0c19kaXIgLyBzYWZlX2pvYl9pZAogICAgCiAgICB1cGxvYWRfZmlsZSA9IGF2YXRhciBvciB2aWRlbwogICAgaWYgbm90IHVwbG9hZF9maWxlOgogICAgICAgIHJhaXNlIGVycm9yX3Jlc3BvbnNlKAogICAgICAgICAgICBzdGF0dXNfY29kZT1zdGF0dXMuSFRUUF80MDBfQkFEX1JFUVVFU1QsCiAgICAgICAgICAgIGVycm9yPSJBdmF0YXIgb3UgdmlkZW8gw6kgb2JyaWdhdMOzcmlvLiIsCiAgICAgICAgICAgIGNvZGU9IkZJTEVfTk9UX0ZPVU5EIgogICAgICAgICkKICAgICAgICAKICAgIGF2YXRhcl9wYXRoID0gc2F2ZV91cGxvYWQodXBsb2FkX2ZpbGUsIGpvYl9kaXIgLyBzYWZlX3VwbG9hZF9uYW1lKHVwbG9hZF9maWxlLmZpbGVuYW1lLCAiYXZhdGFyLmpwZyIpKQogICAgYXVkaW9fcGF0aCA9IHNhdmVfdXBsb2FkKGF1ZGlvLCBqb2JfZGlyIC8gc2FmZV91cGxvYWRfbmFtZShhdWRpby5maWxlbmFtZSwgImF1ZGlvLndhdiIpKQogICAgbG9nZ2VyLmluZm8oIkpvYiAlcyByZWNlYmlkbyB2aWEgdXBsb2FkOiBhdmF0YXIvdmlkZW89JXMgYXVkaW89JXMiLCBqb2JJZCwgYXZhdGFyX3BhdGgsIGF1ZGlvX3BhdGgpCgogICAgdmlkZW9fcGF0aCA9IHJ1bl9nZW5lcmF0aW9uKAogICAgICAgIGpvYl9pZD1zYWZlX2pvYl9pZCwKICAgICAgICBhdmF0YXJfcGF0aD1hdmF0YXJfcGF0aCwKICAgICAgICBhdWRpb19wYXRoPWF1ZGlvX3BhdGgsCiAgICApCiAgICB2aWRlb191cmwgPSBzdHIocmVxdWVzdC51cmxfZm9yKCJkb3dubG9hZF9vdXRwdXQiLCBqb2JfaWQ9c2FmZV9qb2JfaWQsIGZpbGVuYW1lPXZpZGVvX3BhdGgubmFtZSkpCiAgICBsb2dnZXIuaW5mbygiSm9iICVzIGNvbmNsdcOtZG8gdmlhIHVwbG9hZDogJXMiLCBqb2JJZCwgdmlkZW9fcGF0aCkKICAgIHJldHVybiBHZW5lcmF0ZVJlc3BvbnNlKHN1Y2Nlc3M9VHJ1ZSwgcHJvdmlkZXI9Im11c2V0YWxrLXYxNSIsIHZpZGVvX3BhdGg9c3RyKHZpZGVvX3BhdGgpLCB2aWRlb191cmw9dmlkZW9fdXJsKQoKCkBhcHAuZ2V0KCIvb3V0cHV0cy97am9iX2lkfS97ZmlsZW5hbWV9IiwgbmFtZT0iZG93bmxvYWRfb3V0cHV0IikKZGVmIGRvd25sb2FkX291dHB1dChqb2JfaWQ6IHN0ciwgZmlsZW5hbWU6IHN0ciwgXzogTm9uZSA9IERlcGVuZHModmVyaWZ5X2FwaV9rZXkpKSAtPiBGaWxlUmVzcG9uc2U6CiAgICBzYWZlX2pvYl9pZCA9IHNhZmVfcGF0aF9zZWdtZW50KGpvYl9pZCkKICAgIHNhZmVfZmlsZW5hbWUgPSBQYXRoKGZpbGVuYW1lKS5uYW1lCiAgICBvdXRwdXRfcGF0aCA9IHNlcnZpY2Uub3V0cHV0c19kaXIgLyBzYWZlX2pvYl9pZCAvIHNhZmVfZmlsZW5hbWUKICAgIGlmIG5vdCBvdXRwdXRfcGF0aC5leGlzdHMoKSBvciBub3Qgb3V0cHV0X3BhdGguaXNfZmlsZSgpOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oCiAgICAgICAgICAgIHN0YXR1c19jb2RlPXN0YXR1cy5IVFRQXzQwNF9OT1RfRk9VTkQsCiAgICAgICAgICAgIGRldGFpbD17CiAgICAgICAgICAgICAgICAic3VjY2VzcyI6IEZhbHNlLAogICAgICAgICAgICAgICAgInByb3ZpZGVyIjogIm11c2V0YWxrLXYxNSIsCiAgICAgICAgICAgICAgICAiZXJyb3IiOiBmIk91dHB1dCBuw6NvIGVuY29udHJhZG86IHtzYWZlX2ZpbGVuYW1lfSIsCiAgICAgICAgICAgICAgICAiY29kZSI6ICJGSUxFX05PVF9GT1VORCIKICAgICAgICAgICAgfSwKICAgICAgICApCiAgICByZXR1cm4gRmlsZVJlc3BvbnNlKG91dHB1dF9wYXRoLCBtZWRpYV90eXBlPSJ2aWRlby9tcDQiLCBmaWxlbmFtZT1zYWZlX2ZpbGVuYW1lKQo="
service_b64 = "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGdsb2IKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBzaGxleAppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gc3RyaW5nIGltcG9ydCBUZW1wbGF0ZQoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoIm1yY2hpY2tlbi5saXBzeW5jIikKCgpjbGFzcyBHUFVVbmF2YWlsYWJsZUVycm9yKFJ1bnRpbWVFcnJvcik6CiAgICAiIiJSYWlzZWQgd2hlbiBNdXNlVGFsayBpcyBjb25maWd1cmVkIHRvIHJlcXVpcmUgQ1VEQSBidXQgbm8gR1BVIGlzIHZpc2libGUuIiIiCgoKY2xhc3MgTXVzZVRhbGtFeGVjdXRpb25FcnJvcihSdW50aW1lRXJyb3IpOgogICAgIiIiUmFpc2VkIHdoZW4gdGhlIE11c2VUYWxrIHByb2Nlc3MgZXhpdHMgd2l0aCBhIG5vbi16ZXJvIHN0YXR1cyBvciBubyBvdXRwdXQuIiIiCgoKY2xhc3MgTXVzZVRhbGtUaW1lb3V0RXJyb3IoVGltZW91dEVycm9yKToKICAgICIiIlJhaXNlZCB3aGVuIE11c2VUYWxrIGV4Y2VlZHMgdGhlIGNvbmZpZ3VyZWQgdGltZW91dC4iIiIKCgpjbGFzcyBNdXNlVGFsa1NlcnZpY2U6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWxzX2RpcjogUGF0aCwgb3V0cHV0c19kaXI6IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5tb2RlbHNfZGlyID0gbW9kZWxzX2RpcgogICAgICAgIHNlbGYub3V0cHV0c19kaXIgPSBvdXRwdXRzX2RpcgogICAgICAgIHNlbGYubW9kZWxzX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2VsZi5vdXRwdXRzX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGVmIGdwdV9hdmFpbGFibGUoc2VsZikgLT4gYm9vbDoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGltcG9ydCB0b3JjaCAgIyB0eXBlOiBpZ25vcmUKCiAgICAgICAgICAgIGlmIGJvb2wodG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSk6CiAgICAgICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgICAgIFsibnZpZGlhLXNtaSIsICItLXF1ZXJ5LWdwdT1uYW1lIiwgIi0tZm9ybWF0PWNzdixub2hlYWRlciJdLAogICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwKICAgICAgICAgICAgICAgIHRleHQ9VHJ1ZSwKICAgICAgICAgICAgICAgIHRpbWVvdXQ9NSwKICAgICAgICAgICAgICAgIGNoZWNrPUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJldHVybiByZXN1bHQucmV0dXJuY29kZSA9PSAwIGFuZCBib29sKHJlc3VsdC5zdGRvdXQuc3RyaXAoKSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKCiAgICBkZWYgZ2VuZXJhdGUoc2VsZiwgam9iX2lkOiBzdHIsIGF2YXRhcl9wYXRoOiBQYXRoLCBhdWRpb19wYXRoOiBQYXRoKSAtPiBQYXRoOgogICAgICAgIHNlbGYuX3ZhbGlkYXRlX2lucHV0cyhhdmF0YXJfcGF0aD1hdmF0YXJfcGF0aCwgYXVkaW9fcGF0aD1hdWRpb19wYXRoKQogICAgICAgIHNlbGYuX3ZhbGlkYXRlX2dwdSgpCgogICAgICAgIGpvYl9kaXIgPSBzZWxmLm91dHB1dHNfZGlyIC8gam9iX2lkCiAgICAgICAgam9iX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgCiAgICAgICAgdmVyc2lvbiA9IG9zLmdldGVudigiTVVTRVRBTEtfVkVSU0lPTiIsICJ2MTUiKQogICAgICAgIGZpbGVuYW1lID0gZiJtdXNldGFsay17dmVyc2lvbn0tb3V0cHV0Lm1wNCIgaWYgdmVyc2lvbiBlbHNlICJtdXNldGFsay1vdXRwdXQubXA0IgogICAgICAgIG91dHB1dF9wYXRoID0gam9iX2RpciAvIGZpbGVuYW1lCgogICAgICAgIGNvbW1hbmQgPSBzZWxmLl9idWlsZF9jb21tYW5kKAogICAgICAgICAgICBqb2JfaWQ9am9iX2lkLAogICAgICAgICAgICBhdmF0YXJfcGF0aD1hdmF0YXJfcGF0aCwKICAgICAgICAgICAgYXVkaW9fcGF0aD1hdWRpb19wYXRoLAogICAgICAgICAgICBvdXRwdXRfcGF0aD1vdXRwdXRfcGF0aCwKICAgICAgICAgICAgam9iX2Rpcj1qb2JfZGlyLAogICAgICAgICkKICAgICAgICB0aW1lb3V0ID0gaW50KG9zLmdldGVudigiTVVTRVRBTEtfVElNRU9VVF9TRUNPTkRTIiwgIjE4MDAiKSkKICAgICAgICBjd2QgPSBvcy5nZXRlbnYoIk1VU0VUQUxLX1JFUE9fUEFUSCIpIG9yIE5vbmUKCiAgICAgICAgIyBCdWlsZCBjdXN0b20gZW52aXJvbm1lbnQgdG8gcGFzcyBGRk1QRUdfUEFUSCBhbmQgUFlUSE9OUEFUSAogICAgICAgIGVudiA9IG9zLmVudmlyb24uY29weSgpCiAgICAgICAgZmZtcGVnX3BhdGggPSBvcy5nZXRlbnYoIk1VU0VUQUxLX0ZGTVBFR19QQVRIIikKICAgICAgICBpZiBmZm1wZWdfcGF0aDoKICAgICAgICAgICAgZW52WyJGRk1QRUdfUEFUSCJdID0gZmZtcGVnX3BhdGgKICAgICAgICAgICAgIyBBbHNvIHByZXBlbmQgdG8gc3lzdGVtIHBhdGggaWYgbmVlZGVkCiAgICAgICAgICAgIGZmbXBlZ19kaXIgPSBzdHIoUGF0aChmZm1wZWdfcGF0aCkucGFyZW50KQogICAgICAgICAgICBlbnZbIlBBVEgiXSA9IGZmbXBlZ19kaXIgKyBvcy5wYXRoc2VwICsgZW52LmdldCgiUEFUSCIsICIiKQoKICAgICAgICBpZiBjd2Q6CiAgICAgICAgICAgIGVudlsiUFlUSE9OUEFUSCJdID0gZiJ7Y3dkfXtvcy5wYXRoc2VwfXtlbnYuZ2V0KCdQWVRIT05QQVRIJywgJycpfSIKCiAgICAgICAgc3Rkb3V0X2ZpbGUgPSBqb2JfZGlyIC8gInN0ZG91dC5sb2ciCiAgICAgICAgc3RkZXJyX2ZpbGUgPSBqb2JfZGlyIC8gInN0ZGVyci5sb2ciCiAgICAgICAgZXJyb3JfZmlsZSA9IGpvYl9kaXIgLyAiZXJyb3IubG9nIgoKICAgICAgICBsb2dnZXIuaW5mbygiRXhlY3V0YW5kbyBNdXNlVGFsayBwYXJhIGpvYiAlczogJXMiLCBqb2JfaWQsICIgIi5qb2luKGNvbW1hbmQpKQogICAgICAgIHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgICAgIGNvbW1hbmQsCiAgICAgICAgICAgICAgICBjd2Q9Y3dkLAogICAgICAgICAgICAgICAgZW52PWVudiwKICAgICAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgICAgICB0ZXh0PVRydWUsCiAgICAgICAgICAgICAgICB0aW1lb3V0PXRpbWVvdXQsCiAgICAgICAgICAgICAgICBjaGVjaz1GYWxzZSwKICAgICAgICAgICAgKQogICAgICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkIGFzIGV4YzoKICAgICAgICAgICAgZXJyb3JfZmlsZS53cml0ZV90ZXh0KGYiVGltZW91dEV4cGlyZWQ6IE11c2VUYWxrIGV4Y2VkZXUgbyB0aW1lb3V0IGRlIHt0aW1lb3V0fXMuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICAgICAgcmFpc2UgTXVzZVRhbGtUaW1lb3V0RXJyb3IoZiJNdXNlVGFsayBleGNlZGV1IG8gdGltZW91dCBkZSB7dGltZW91dH1zIHBhcmEgbyBqb2Ige2pvYl9pZH0uIikgZnJvbSBleGMKCiAgICAgICAgZWxhcHNlZCA9IHRpbWUubW9ub3RvbmljKCkgLSBzdGFydGVkCiAgICAgICAgCiAgICAgICAgIyBTYXZlIHN0ZG91dC9zdGRlcnIgbG9ncwogICAgICAgIHN0ZG91dF9maWxlLndyaXRlX3RleHQocmVzdWx0LnN0ZG91dCBvciAiIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICBzdGRlcnJfZmlsZS53cml0ZV90ZXh0KHJlc3VsdC5zdGRlcnIgb3IgIiIsIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgICAgIGlmIHJlc3VsdC5zdGRvdXQ6CiAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJNdXNlVGFsayBzdGRvdXQgam9iICVzOlxuJXMiLCBqb2JfaWQsIHJlc3VsdC5zdGRvdXRbLTIwMDA6XSkKICAgICAgICBpZiByZXN1bHQuc3RkZXJyOgogICAgICAgICAgICBsb2dnZXIud2FybmluZygiTXVzZVRhbGsgc3RkZXJyIGpvYiAlczpcbiVzIiwgam9iX2lkLCByZXN1bHQuc3RkZXJyWy0yMDAwOl0pCgogICAgICAgIGlmIHJlc3VsdC5yZXR1cm5jb2RlICE9IDA6CiAgICAgICAgICAgIHRyYWNlYmFja19zbmlwcGV0ID0gKHJlc3VsdC5zdGRlcnIgb3IgIiIpWy0yMDAwOl0KICAgICAgICAgICAgZXJyb3JfbXNnID0gZiJNdXNlVGFsayBmYWxob3UgY29tIGPDs2RpZ28ge3Jlc3VsdC5yZXR1cm5jb2RlfTpcbnt0cmFjZWJhY2tfc25pcHBldH0iCiAgICAgICAgICAgIGVycm9yX2ZpbGUud3JpdGVfdGV4dChmIlJldHVybiBjb2RlOiB7cmVzdWx0LnJldHVybmNvZGV9XG5cblN0ZGVycjpcbntyZXN1bHQuc3RkZXJyfVxuXG5TdGRvdXQ6XG57cmVzdWx0LnN0ZG91dH0iLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICByYWlzZSBNdXNlVGFsa0V4ZWN1dGlvbkVycm9yKGVycm9yX21zZykKCiAgICAgICAgZmluYWxfcGF0aCA9IHNlbGYuX3Jlc29sdmVfb3V0cHV0KGpvYl9kaXI9am9iX2RpciwgZXhwZWN0ZWRfcGF0aD1vdXRwdXRfcGF0aCkKICAgICAgICBsb2dnZXIuaW5mbygiTXVzZVRhbGsgZmluYWxpem91IGpvYiAlcyBlbSAlLjFmczogJXMiLCBqb2JfaWQsIGVsYXBzZWQsIGZpbmFsX3BhdGgpCiAgICAgICAgcmV0dXJuIGZpbmFsX3BhdGgKCiAgICBkZWYgX3ZhbGlkYXRlX2lucHV0cyhzZWxmLCBhdmF0YXJfcGF0aDogUGF0aCwgYXVkaW9fcGF0aDogUGF0aCkgLT4gTm9uZToKICAgICAgICBpZiBub3QgYXZhdGFyX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiQXJxdWl2byBkZSBhdmF0YXIgbsOjbyBlbmNvbnRyYWRvOiB7YXZhdGFyX3BhdGh9IikKICAgICAgICBpZiBub3QgYXVkaW9fcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJBcnF1aXZvIGRlIMOhdWRpbyBuw6NvIGVuY29udHJhZG86IHthdWRpb19wYXRofSIpCgogICAgZGVmIF92YWxpZGF0ZV9ncHUoc2VsZikgLT4gTm9uZToKICAgICAgICByZXF1aXJlX2dwdSA9IG9zLmdldGVudigiTVVTRVRBTEtfUkVRVUlSRV9HUFUiLCAidHJ1ZSIpLmxvd2VyKCkgaW4geyIxIiwgInRydWUiLCAieWVzIiwgInNpbSJ9CiAgICAgICAgaWYgcmVxdWlyZV9ncHUgYW5kIG5vdCBzZWxmLmdwdV9hdmFpbGFibGUoKToKICAgICAgICAgICAgcmFpc2UgR1BVVW5hdmFpbGFibGVFcnJvcigiR1BVL0NVREEgaW5kaXNwb27DrXZlbCBwYXJhIGV4ZWN1dGFyIE11c2VUYWxrLiIpCgogICAgZGVmIF9idWlsZF9jb21tYW5kKHNlbGYsIGpvYl9pZDogc3RyLCBhdmF0YXJfcGF0aDogUGF0aCwgYXVkaW9fcGF0aDogUGF0aCwgb3V0cHV0X3BhdGg6IFBhdGgsIGpvYl9kaXI6IFBhdGgpIC0+IGxpc3Rbc3RyXToKICAgICAgICB0ZW1wbGF0ZSA9IG9zLmdldGVudigiTVVTRVRBTEtfQ09NTUFORF9URU1QTEFURSIpCiAgICAgICAgaWYgdGVtcGxhdGU6CiAgICAgICAgICAgIHJlbmRlcmVkID0gVGVtcGxhdGUodGVtcGxhdGUpLnNhZmVfc3Vic3RpdHV0ZSgKICAgICAgICAgICAgICAgIGpvYklkPWpvYl9pZCwKICAgICAgICAgICAgICAgIGF2YXRhclBhdGg9c3RyKGF2YXRhcl9wYXRoKSwKICAgICAgICAgICAgICAgIGF1ZGlvUGF0aD1zdHIoYXVkaW9fcGF0aCksCiAgICAgICAgICAgICAgICBvdXRwdXRQYXRoPXN0cihvdXRwdXRfcGF0aCksCiAgICAgICAgICAgICAgICBqb2JEaXI9c3RyKGpvYl9kaXIpLAogICAgICAgICAgICAgICAgbW9kZWxzRGlyPXN0cihzZWxmLm1vZGVsc19kaXIpLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJldHVybiBzaGxleC5zcGxpdChyZW5kZXJlZCkKCiAgICAgICAgcmVwb19wYXRoID0gb3MuZ2V0ZW52KCJNVVNFVEFMS19SRVBPX1BBVEgiKQogICAgICAgIGlmIG5vdCByZXBvX3BhdGg6CiAgICAgICAgICAgIHJhaXNlIE11c2VUYWxrRXhlY3V0aW9uRXJyb3IoCiAgICAgICAgICAgICAgICAiQ29uZmlndXJlIE1VU0VUQUxLX1JFUE9fUEFUSCBvdSBNVVNFVEFMS19DT01NQU5EX1RFTVBMQVRFIHBhcmEgYXBvbnRhciBwYXJhIHVtYSBpbnN0YWxhw6fDo28gZG8gTXVzZVRhbGsuIgogICAgICAgICAgICApCgogICAgICAgIHJlcG8gPSBQYXRoKHJlcG9fcGF0aCkKICAgICAgICBpbmZlcmVuY2Vfc2NyaXB0ID0gUGF0aChvcy5nZXRlbnYoIk1VU0VUQUxLX0lORkVSRU5DRV9TQ1JJUFQiLCByZXBvIC8gInNjcmlwdHMiIC8gImluZmVyZW5jZS5weSIpKQogICAgICAgIGlmIG5vdCBpbmZlcmVuY2Vfc2NyaXB0LmV4aXN0cygpOgogICAgICAgICAgICByYWlzZSBNdXNlVGFsa0V4ZWN1dGlvbkVycm9yKGYiU2NyaXB0IGRlIGluZmVyw6puY2lhIGRvIE11c2VUYWxrIG7Do28gZW5jb250cmFkbzoge2luZmVyZW5jZV9zY3JpcHR9IikKCiAgICAgICAgY29uZmlnX3BhdGggPSBzZWxmLl93cml0ZV9pbmZlcmVuY2VfY29uZmlnKAogICAgICAgICAgICBqb2JfaWQ9am9iX2lkLAogICAgICAgICAgICBhdmF0YXJfcGF0aD1hdmF0YXJfcGF0aCwKICAgICAgICAgICAgYXVkaW9fcGF0aD1hdWRpb19wYXRoLAogICAgICAgICAgICBqb2JfZGlyPWpvYl9kaXIsCiAgICAgICAgKQogICAgICAgIAogICAgICAgIHB5dGhvbiA9IG9zLmdldGVudigiTVVTRVRBTEtfUFlUSE9OIikgb3Igc3lzLmV4ZWN1dGFibGUKICAgICAgICB2ZXJzaW9uID0gb3MuZ2V0ZW52KCJNVVNFVEFMS19WRVJTSU9OIiwgInYxNSIpCiAgICAgICAgdW5ldF9tb2RlbF9wYXRoID0gb3MuZ2V0ZW52KCJNVVNFVEFMS19VTkVUX01PREVMX1BBVEgiLCAibW9kZWxzL211c2V0YWxrVjE1L3VuZXQucHRoIikKICAgICAgICB1bmV0X2NvbmZpZyA9IG9zLmdldGVudigiTVVTRVRBTEtfVU5FVF9DT05GSUciLCAibW9kZWxzL211c2V0YWxrVjE1L211c2V0YWxrLmpzb24iKQogICAgICAgIAogICAgICAgIHJldHVybiBbCiAgICAgICAgICAgIHB5dGhvbiwgCiAgICAgICAgICAgIHN0cihpbmZlcmVuY2Vfc2NyaXB0KSwgCiAgICAgICAgICAgICItLWluZmVyZW5jZV9jb25maWciLCBzdHIoY29uZmlnX3BhdGgpLAogICAgICAgICAgICAiLS12ZXJzaW9uIiwgdmVyc2lvbiwKICAgICAgICAgICAgIi0tdW5ldF9tb2RlbF9wYXRoIiwgdW5ldF9tb2RlbF9wYXRoLAogICAgICAgICAgICAiLS11bmV0X2NvbmZpZyIsIHVuZXRfY29uZmlnCiAgICAgICAgXQoKICAgIGRlZiBfd3JpdGVfaW5mZXJlbmNlX2NvbmZpZyhzZWxmLCBqb2JfaWQ6IHN0ciwgYXZhdGFyX3BhdGg6IFBhdGgsIGF1ZGlvX3BhdGg6IFBhdGgsIGpvYl9kaXI6IFBhdGgpIC0+IFBhdGg6CiAgICAgICAgIyBNdXNlVGFsayB1c2VzIFlBTUwgY29uZmlncyBzaGFwZWQgYXMgdGFzayBtYXBzLiBLZWVwIHRoaXMgZmlsZSBtaW5pbWFsIGFuZAogICAgICAgICMgbGV0IHRoZSBjaGVja2VkLW91dCBNdXNlVGFsayByZXBvc2l0b3J5IG93biBtb2RlbC9jaGVja3BvaW50IHNldHRpbmdzLgogICAgICAgIGNvbmZpZ19wYXRoID0gam9iX2RpciAvICJtdXNldGFsay1pbmZlcmVuY2UueWFtbCIKICAgICAgICBjb25maWdfcGF0aC53cml0ZV90ZXh0KAogICAgICAgICAgICAidGFza18wOlxuIgogICAgICAgICAgICBmIiAgdmlkZW9fcGF0aDogJ3thdmF0YXJfcGF0aC5hc19wb3NpeCgpfSdcbiIKICAgICAgICAgICAgZiIgIGF1ZGlvX3BhdGg6ICd7YXVkaW9fcGF0aC5hc19wb3NpeCgpfSdcbiIKICAgICAgICAgICAgZiIgIHJlc3VsdF9kaXI6ICd7am9iX2Rpci5hc19wb3NpeCgpfSdcbiIKICAgICAgICAgICAgZiIgIHJlc3VsdF9uYW1lOiAne2pvYl9pZH0nXG4iLAogICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICkKICAgICAgICByZXR1cm4gY29uZmlnX3BhdGgKCiAgICBkZWYgX3Jlc29sdmVfb3V0cHV0KHNlbGYsIGpvYl9kaXI6IFBhdGgsIGV4cGVjdGVkX3BhdGg6IFBhdGgpIC0+IFBhdGg6CiAgICAgICAgaWYgZXhwZWN0ZWRfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgcmV0dXJuIGV4cGVjdGVkX3BhdGgKCiAgICAgICAgY2FuZGlkYXRlcyA9IFtQYXRoKHApIGZvciBwIGluIGdsb2IuZ2xvYihzdHIoam9iX2RpciAvICIqKiIgLyAiKi5tcDQiKSwgcmVjdXJzaXZlPVRydWUpXQogICAgICAgIHJlcG9fcGF0aCA9IG9zLmdldGVudigiTVVTRVRBTEtfUkVQT19QQVRIIikKICAgICAgICBpZiByZXBvX3BhdGg6CiAgICAgICAgICAgIHJlcG9fcmVzdWx0cyA9IFBhdGgocmVwb19wYXRoKSAvICJyZXN1bHRzIgogICAgICAgICAgICBjYW5kaWRhdGVzLmV4dGVuZChQYXRoKHApIGZvciBwIGluIGdsb2IuZ2xvYihzdHIocmVwb19yZXN1bHRzIC8gIioqIiAvICIqLm1wNCIpLCByZWN1cnNpdmU9VHJ1ZSkpCgogICAgICAgIGV4aXN0aW5nID0gW2NhbmRpZGF0ZSBmb3IgY2FuZGlkYXRlIGluIGNhbmRpZGF0ZXMgaWYgY2FuZGlkYXRlLmV4aXN0cygpXQogICAgICAgIGlmIG5vdCBleGlzdGluZzoKICAgICAgICAgICAgcmFpc2UgTXVzZVRhbGtFeGVjdXRpb25FcnJvcigiTXVzZVRhbGsgdGVybWlub3UsIG1hcyBuZW5odW0gYXJxdWl2byAubXA0IGRlIHNhw61kYSBmb2kgZW5jb250cmFkby4iKQoKICAgICAgICBsYXRlc3QgPSBtYXgoZXhpc3RpbmcsIGtleT1sYW1iZGEgcGF0aDogcGF0aC5zdGF0KCkuc3RfbXRpbWUpCiAgICAgICAgaWYgbGF0ZXN0ICE9IGV4cGVjdGVkX3BhdGg6CiAgICAgICAgICAgIHNodXRpbC5jb3B5MihsYXRlc3QsIGV4cGVjdGVkX3BhdGgpCiAgICAgICAgcmV0dXJuIGV4cGVjdGVkX3BhdGgK"

app_path = WORK_DIR / 'app.py'
service_path = WORK_DIR / 'musetalk_service.py'

app_path.write_bytes(base64.b64decode(app_b64))
service_path.write_bytes(base64.b64decode(service_b64))

print('Arquivos escritos com sucesso:')
print(' -', app_path)
print(' -', service_path)


In [ ]:
# 6. Iniciar FastAPI e expor via Cloudflare Tunnel
import os, queue, re, stat, threading, subprocess, time, requests, sys
PYTHON_EXE = os.environ.get('PYTHON') or sys.executable

cloudflared = WORK_DIR / 'cloudflared'
if not cloudflared.exists():
    print("Baixando binário do Cloudflare Tunnel...")
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', str(cloudflared)], check=True)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

# Finaliza instâncias anteriores para liberar porta se o notebook for reiniciado
for name in ['server_proc', 'tunnel_proc']:
    proc = globals().get(name)
    if proc and proc.poll() is None:
        print(f"Finalizando processo antigo: {name}")
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()

print("Iniciando serviço FastAPI com Uvicorn...")
os.environ['PYTHONPATH'] = f"{str(WORK_DIR)}:{os.environ.get('PYTHONPATH', '')}"
server_proc = subprocess.Popen(
    [PYTHON_EXE, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', str(PORT), '--proxy-headers'],
    cwd=str(WORK_DIR),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

def stream_logs(prefix, proc):
    for line in proc.stdout:
        print(prefix, line, end='', flush=True)
threading.Thread(target=stream_logs, args=('[uvicorn]', server_proc), daemon=True).start()

time.sleep(5)
try:
    resp = requests.get(f'http://127.0.0.1:{PORT}/health', timeout=15)
    print('Smoke Test Health Local:', resp.status_code, resp.json())
except Exception as e:
    print('Erro no smoke test local de saúde:', e)

print("Iniciando túnel Cloudflare...")
tunnel_proc = subprocess.Popen(
    [str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

q = queue.Queue()
def capture_tunnel():
    for line in tunnel_proc.stdout:
        print('[cloudflared]', line, end='', flush=True)
        q.put(line)
threading.Thread(target=capture_tunnel, daemon=True).start()

PUBLIC_URL = None
pattern = re.compile(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com')
deadline = time.time() + 90
while time.time() < deadline and PUBLIC_URL is None:
    try:
        line = q.get(timeout=2)
    except queue.Empty:
        continue
    match = pattern.search(line)
    if match:
        PUBLIC_URL = match.group(0)
if not PUBLIC_URL:
    raise RuntimeError('Não consegui obter URL pública do cloudflared.')

print('\n=== CONFIGURE NO .env.local DO MRCHICKEN ===')
print('LIPSYNC_ENGINE=musetalk-v15')
print(f'LIPSYNC_API_URL={PUBLIC_URL}')
print(f'LIPSYNC_API_KEY={API_KEY}')
print('LIPSYNC_TRANSFER_MODE=upload')
print('LIPSYNC_TIMEOUT_MS=1800000')
print('\nEndpoint público:', PUBLIC_URL)
